# 02 — Carga e inspección

La primera celda de cualquier notebook de análisis hace estas dos cosas: cargar los datos e inspeccionarlos. Inspeccionar no es leer los datos visualmente, es ejecutar funciones que producen un diagnóstico estructurado.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'


## read_csv — parámetros esenciales

La mayoría de datasets del mundo real requieren al menos un parámetro extra en `read_csv`. Los defaults no siempre son los correctos.

In [2]:
# Parámetros más usados en la práctica
df = pd.read_csv(
    TRAIN,
    # encoding='utf-8',          # codificación de caracteres (default)
    # encoding='latin-1',        # para archivos con caracteres especiales no UTF-8
    # sep=';',                   # separador (default ',')
    # decimal=',',               # decimal europeo
    # thousands='.',             # separador de miles
    low_memory=False,            # infiere tipos sobre el archivo completo, no por chunks
    # usecols=['Sales', 'Region'],  # cargar solo columnas necesarias
    # nrows=1000,                # cargar solo N filas (útil para exploración)
    # skiprows=1,                # saltar filas al inicio
    # parse_dates=['Order Date'], # convertir a datetime al cargar
    # dtype={'Postal Code': str}, # forzar tipo por columna
)
print(f'Cargado: {df.shape}')


Cargado: (9800, 18)


In [3]:
# Airbnb necesita latin-1 y tiene 33 columnas — ejemplo de caso real
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
print(f'Airbnb: {air.shape}')
print(f'Memoria: {air.memory_usage(deep=True).sum() / 1024**2:.1f} MB')


Airbnb: (279712, 33)


Memoria: 334.5 MB


## info() — el diagnóstico más completo

`info()` muestra en una sola llamada: número de filas, tipos de cada columna, y cuántos valores no nulos tiene cada una. Es la primera función que se ejecuta siempre.

In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

In [5]:
# Para datasets grandes, show_counts=True asegura que cuente los no-nulos
# memory_usage='deep' calcula el uso real de memoria (más lento pero preciso)
air.info(memory_usage='deep', show_counts=True)


<class 'pandas.DataFrame'>
RangeIndex: 279712 entries, 0 to 279711
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   listing_id                   279712 non-null  int64  
 1   name                         279537 non-null  str    
 2   host_id                      279712 non-null  int64  
 3   host_since                   279547 non-null  str    
 4   host_location                278872 non-null  str    
 5   host_response_time           150930 non-null  str    
 6   host_response_rate           150930 non-null  float64
 7   host_acceptance_rate         166625 non-null  float64
 8   host_is_superhost            279547 non-null  str    
 9   host_total_listings_count    279547 non-null  float64
 10  host_has_profile_pic         279547 non-null  str    
 11  host_identity_verified       279547 non-null  str    
 12  neighbourhood                279712 non-null  str    
 13  district  

## describe() — estadísticas descriptivas

In [6]:
# Por defecto solo incluye columnas numéricas
print(df.describe().round(2))
print()

# include='object' para columnas de texto
print(df.describe(include='object'))


        Row ID  Postal Code     Sales
count  9800.00      9789.00   9800.00
mean   4900.50     55273.32    230.77
std    2829.16     32041.22    626.65
min       1.00      1040.00      0.44
25%    2450.75     23223.00     17.25
50%    4900.50     58103.00     54.49
75%    7350.25     90008.00    210.60
max    9800.00     99301.00  22638.48

              Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
count             9800        9800        9800            9800        9800   
unique            4922        1230        1326               4         793   
top     CA-2018-100111  05/09/2017  26/09/2018  Standard Class    WB-21850   
freq                14          38          34            5859          35   

        Customer Name   Segment        Country           City       State  \
count            9800      9800           9800           9800        9800   
unique            793         3              1            529          49   
top     William Brown  Consumer  Un

C:\Users\alefe\AppData\Local\Temp\ipykernel_16732\2170636226.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include='object'))


## value_counts, nunique, y sample

In [7]:
# value_counts — frecuencia de cada valor, ordenada de mayor a menor
print(df['Region'].value_counts())
print()

# normalize=True para proporciones en vez de conteos
print(df['Category'].value_counts(normalize=True).round(3))
print()

# nunique — número de valores únicos por columna
print(df.nunique().sort_values())


Region
West       3140
East       2785
Central    2277
South      1598
Name: count, dtype: int64

Category
Office Supplies    0.603
Furniture          0.212
Technology         0.185
Name: proportion, dtype: float64

Country             1
Segment             3
Category            3
Ship Mode           4
Region              4
Sub-Category       17
State              49
City              529
Postal Code       626
Customer Name     793
Customer ID       793
Order Date       1230
Ship Date        1326
Product Name     1849
Product ID       1861
Order ID         4922
Sales            5757
Row ID           9800
dtype: int64


In [8]:
# sample() — muestra aleatoria, útil para inspeccionar filas reales
print(df.sample(5, random_state=42))
print()

# head/tail
print(df.tail(3))


      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
532      533  US-2018-129441  07/09/2018  11/09/2018  Standard Class   
872      873  CA-2015-148488  10/12/2015  15/12/2015  Standard Class   
1149    1150  CA-2016-112452  04/04/2016  04/04/2016        Same Day   
2287    2288  US-2018-112928  01/06/2018  05/06/2018    Second Class   
4038    4039  CA-2015-110786  29/12/2015  02/01/2016  Standard Class   

     Customer ID    Customer Name    Segment        Country           City  \
532     JC-15340  Jasper Cacioppo   Consumer  United States    Los Angeles   
872     SM-20005   Sally Matthias   Consumer  United States  New York City   
1149    NC-18340      Nat Carroll   Consumer  United States        Lansing   
2287    BB-10990  Barry Blumstein  Corporate  United States         Toledo   
4038    AJ-10795  Anthony Johnson  Corporate  United States  San Francisco   

           State  Postal Code   Region       Product ID         Category  \
532   California      

## Diagnóstico de nulos

In [9]:
# isnull().sum() — conteo de nulos por columna
nulos = df.isnull().sum()
print('Columnas con nulos:')
print(nulos[nulos > 0])
print()

# Porcentaje — dividir entre el total de filas
pct_nulos = (df.isnull().mean() * 100).round(2)
print('Porcentaje de nulos:')
print(pct_nulos[pct_nulos > 0])


Columnas con nulos:
Postal Code    11
dtype: int64



Porcentaje de nulos:
Postal Code    0.11
dtype: float64


---
## Resumen

| Función | Qué muestra |
|---------|-------------|
| `df.info()` | Tipos, non-null counts, memoria |
| `df.describe()` | Estadísticas numéricas |
| `df.describe(include='object')` | Stats para columnas de texto |
| `df['col'].value_counts()` | Frecuencia de cada valor |
| `df.nunique()` | Valores únicos por columna |
| `df.sample(n)` | n filas aleatorias |
| `df.isnull().sum()` | Nulos por columna |
